# FactIndic interactive summary and evidence check

Paste one **Roman-script Hinglish** article or passage. This live Colab demo generates an English summary, splits it into factual claims, and shows each claim beside source evidence.

## Run it

1. Choose **Runtime → Change runtime type → T4 GPU**.
2. Run all cells from the top. The first run downloads public model weights.
3. Paste Roman-script Hinglish, select a Qwen3 size and prompt, then click **Summarize and check claims**.
4. Follow the **Run progress** tracker beneath the controls. The green tick beside a Colab cell means the cell finished setting up the controls; model work begins only after you click the button and is tracked separately. If the claim checker returns malformed JSON, it retries once; if the retry also fails, the summary is preserved and marked unchecked.
5. Review the claim table and quoted evidence. Use the button to download a JSON report.

This is an **AI-assisted research demo**, not a factuality certificate. The selected Qwen3 model generates and checks its own summary. Evidence quotes are verified against the exact source text shown to the model, but quote presence does not prove logical support and the AI judgment can still be wrong. A claim with no quote or an unverifiable quote is labeled explicitly; number/date mismatches trigger one correction retry; if there is exactly one full date in both source and summary, a remaining date mismatch is replaced from the source and visibly disclosed. Other remaining numeric mismatches mark the summary as an unsafe draft. No factuality probability is invented. Human validation was skipped for this project version.

The fixed 30-source, 120-summary comparison remains in [`factindic_final_open_hinglish_colab.ipynb`](https://colab.research.google.com/github/Sirohi0/factindic/blob/main/factindic_final_open_hinglish_colab.ipynb). This separate notebook is the live single-input demonstration. Inputs and results stay in the Colab runtime unless you download the report; the notebook does not write them to Drive or the public repository.


## 1. Install the inference tools

Use a GPU runtime before running this cell. It downloads models for inference and does not train them.

In [ ]:
%pip -q install "transformers>=4.51.3,<5" accelerate sentencepiece ipywidgets


## 2. Initialize the GPU runtime

In [ ]:
import ast
import gc
import hashlib
import json
import re
import time
import unicodedata
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import torch
from IPython.display import clear_output, display, Markdown
from google.colab import files, output
from transformers import AutoModelForCausalLM, AutoTokenizer

output.enable_custom_widget_manager()
assert torch.cuda.is_available(), "Choose Runtime → Change runtime type → T4 GPU, then rerun this cell."
print("GPU:", torch.cuda.get_device_name(0))

MODEL_IDS = {
    "Qwen3 1.7B": "Qwen/Qwen3-1.7B",
    "Qwen3 4B": "Qwen/Qwen3-4B",
}
MAX_SOURCE_TOKENS = 2048
SUMMARY_TOKEN_LIMIT = 192
CLAIM_CHECK_TOKEN_LIMIT = 768
_loaded_model_key = None
_model = None
_tokenizer = None
_last_result = None

def load_model(model_key):
    global _loaded_model_key, _model, _tokenizer
    if _loaded_model_key == model_key and _model is not None:
        return _model, _tokenizer
    if _model is not None:
        del _model, _tokenizer
        _model, _tokenizer = None, None
        gc.collect()
        torch.cuda.empty_cache()
    model_id = MODEL_IDS[model_key]
    print(f"Loading {model_id} …")
    _tokenizer = AutoTokenizer.from_pretrained(model_id)
    _model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.float16, device_map="auto"
    ).eval()
    # Qwen configs often carry sampling defaults. Set neutral values for greedy
    # decoding so Transformers does not emit misleading ignored-flag warnings.
    _model.generation_config.do_sample = False
    _model.generation_config.temperature = 1.0
    _model.generation_config.top_p = 1.0
    _model.generation_config.top_k = 50
    _loaded_model_key = model_key
    return _model, _tokenizer

def chat_generate(model, tokenizer, user_prompt, max_new_tokens):
    messages = [{"role": "user", "content": user_prompt}]
    encoded = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt", enable_thinking=False
    ).to(model.device)
    input_tokens = int(encoded["input_ids"].shape[-1])
    started = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(
            **encoded, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1,
            repetition_penalty=1.03, no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
        )
    elapsed = time.perf_counter() - started
    new_ids = output_ids[0][input_tokens:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip(), int(len(new_ids)), elapsed

def make_summary(source_used, prompt_style, model, tokenizer, numeric_retry=False):
    shared = (
        "Translate the meaning of this Roman-script Hinglish source into a concise English "
        "summary of at most two sentences. Keep the main event and the exact people, places, "
        "numbers, dates, attribution, and outcome stated in the source. Preserve negation, "
        "warnings, uncertainty, and directions exactly: do not reverse whether people should "
        "approach, avoid, leave, or remain in a place. Do not add advice, causes, identities, "
        "or other details. If a detail is unclear, leave it out. Return only the summary."
    )
    if prompt_style == "Factuality-aware":
        instruction = (
            shared + " Before answering, check that every clause is directly supported by "
            "the source and that no translation changed who did what or what people were told."
        )
    else:
        instruction = shared
    if numeric_retry:
        instruction += (
            " The previous draft changed a date or number. For this retry, copy every "
            "digit sequence and date exactly as written in the source. Do not calculate, "
            "reformat, or guess a number or date; omit it if uncertain."
        )
    prompt = f"{instruction}\n\nSource text:\n{source_used}\n\nEnglish summary:"
    summary, tokens, seconds = chat_generate(model, tokenizer, prompt, SUMMARY_TOKEN_LIMIT)
    return summary, tokens, seconds, tokens >= SUMMARY_TOKEN_LIMIT

def normalize_quote(text):
    return " ".join(unicodedata.normalize("NFKC", str(text)).split()).casefold()

def parse_json_object(text):
    """Extract and parse one object; tolerate fences, prose and common loose JSON."""
    text = str(text).strip()
    start = text.find("{")
    if start < 0:
        raise ValueError("The model did not return a JSON object.")

    # Isolate the first balanced object, respecting both quote styles and escapes.
    stack = []
    quote = None
    escaped = False
    end = None
    pairs = {"}": "{", "]": "["}
    for index in range(start, len(text)):
        char = text[index]
        if quote:
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == quote:
                quote = None
            continue
        if char in {"\"", "'"}:
            quote = char
        elif char in "{[":
            stack.append(char)
        elif char in "}]":
            if not stack or stack[-1] != pairs[char]:
                break
            stack.pop()
            if not stack:
                end = index + 1
                break
    if end is None:
        # Let the regular parsers produce a useful failure on truncated objects.
        candidate = text[start:]
    else:
        candidate = text[start:end]

    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        pass

    # Tolerate trailing commas that are invalid in strict JSON.
    cleaned = re.sub(r",\s*([}\]])", r"\1", candidate)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # Some model responses use Python dict notation with single-quoted strings.
    try:
        return ast.literal_eval(cleaned)
    except (ValueError, SyntaxError):
        pass

    # Last common case: unquoted object keys. Quote only key-like tokens after
    # an opening brace or comma, then try strict JSON and Python literals again.
    quoted_keys = re.sub(r"([{,]\s*)([A-Za-z_][A-Za-z0-9_]*)\s*:", r'\1"\2":', cleaned)
    try:
        return json.loads(quoted_keys)
    except json.JSONDecodeError:
        try:
            return ast.literal_eval(quoted_keys)
        except (ValueError, SyntaxError) as error:
            raise ValueError("The model response is not a parseable claims object.") from error

def parse_claim_payload(text):
    parsed = parse_json_object(text)
    if not isinstance(parsed, dict) or not isinstance(parsed.get("claims"), list):
        raise ValueError("The model response is missing a JSON claims list.")
    return parsed

def check_claims(source_used, summary, model, tokenizer):
    prompt = (
        "You are checking whether a summary is supported by its source. Split it into up to "
        "12 short, atomic factual claims. For each claim choose exactly one status: supported "
        "(source explicitly supports it), contradicted (source explicitly says something "
        "incompatible), not_found (source does not state it), or unclear (source wording does "
        "not let you decide). For supported or contradicted claims, give one short evidence "
        "quote copied exactly from the source. For not_found or unclear claims, use an empty "
        "evidence string. If there are no factual claims to check, return an empty claims list. "
        "Never invent or rewrite a quote. Do not provide numerical probabilities or confidence "
        "scores. Return JSON only in this shape: "
        '{"claims":[{"claim":"...","status":"supported|contradicted|not_found|unclear",'
        '"evidence":"exact source quote or empty"}]}\n\n'
        f"SOURCE:\n{source_used}\n\nSUMMARY:\n{summary}\n"
    )
    raw, tokens, seconds = chat_generate(model, tokenizer, prompt, CLAIM_CHECK_TOKEN_LIMIT)
    try:
        parsed = parse_claim_payload(raw)
    except ValueError:
        # Retry once with a format-only correction. Keep the original source and
        # summary in context so the model cannot silently change what it checks.
        retry_prompt = (
            prompt + "\nFORMAT RETRY: Your previous response was invalid JSON. "
            "Return exactly one valid JSON object using double-quoted keys and strings, "
            "with no markdown, comments, trailing commas, or text before/after it. "
            "Keep the same factual judgments; do not invent or rewrite evidence.\n"
            "Previous response to correct:\n" + raw
        )
        retry_raw, retry_tokens, retry_seconds = chat_generate(
            model, tokenizer, retry_prompt, CLAIM_CHECK_TOKEN_LIMIT
        )
        tokens += retry_tokens
        seconds += retry_seconds
        raw = retry_raw
        try:
            parsed = parse_claim_payload(raw)
        except ValueError as error:
            raise ValueError(
                "The claim checker returned malformed JSON twice. The summary can still be "
                "shown, but no factuality conclusion is available."
            ) from error
    raw_claims = parsed["claims"]
    if not raw_claims:
        return [], tokens, seconds, raw
    allowed = {"supported", "contradicted", "not_found", "unclear"}
    normalized_source = normalize_quote(source_used)
    summary_numbers = set(numeric_tokens(summary))
    results = []
    for item in raw_claims[:12]:
        claim = str(item.get("claim", "")).strip()
        status = str(item.get("status", "unclear")).strip().lower()
        evidence = str(item.get("evidence", "")).strip()
        if not claim:
            continue
        if status not in allowed:
            status = "unclear"
        claim_numbers_not_in_summary = sorted(set(numeric_tokens(claim)) - summary_numbers)
        quote_found = bool(evidence) and normalize_quote(evidence) in normalized_source
        if claim_numbers_not_in_summary:
            # The checker added a number/date absent from the summary it was asked to split.
            # Do not treat any evidence judgment for this mismatched claim as trustworthy.
            status, evidence, quote_found = "checker_mismatch", "", False
        elif status in {"supported", "contradicted"} and not evidence:
            status = "no_evidence_found"
        elif status in {"supported", "contradicted"} and not quote_found:
            status, evidence = "evidence_not_verified", ""
        if status in {"not_found", "unclear", "no_evidence_found"}:
            evidence, quote_found = "", False
        results.append({
            "claim": claim, "assessment": status, "source_evidence": evidence,
            "quote_found_in_model_input": bool(quote_found),
            "claim_numbers_not_in_summary": claim_numbers_not_in_summary
        })
    if not results:
        raise ValueError("No usable claims returned. Rerun or inspect the summary manually.")
    return results, tokens, seconds, raw

def numeric_tokens(text):
    """Return digit sequences so changed dates and figures can be flagged."""
    return re.findall(r"(?<![\w])\d+(?:[.,]\d+)*(?![\w])", str(text))

def unsupported_numeric_tokens(source_text, summary):
    source_values = set(numeric_tokens(source_text))
    return sorted(set(value for value in numeric_tokens(summary) if value not in source_values))

_MONTH_ALIASES = {
    "jan": 1, "january": 1, "feb": 2, "february": 2, "mar": 3, "march": 3,
    "apr": 4, "april": 4, "may": 5, "jun": 6, "june": 6, "jul": 7,
    "july": 7, "aug": 8, "august": 8, "sep": 9, "sept": 9, "september": 9,
    "oct": 10, "october": 10, "nov": 11, "november": 11, "dec": 12, "december": 12,
}
_DATE_PATTERN = re.compile(
    r"\b(?:(?P<day>\d{1,2})\s+(?P<month>January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)\s+(?P<year>\d{2,4})|"
    r"(?P<month_first>January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)\s+(?P<day_first>\d{1,2})(?:st|nd|rd|th)?[,]?\s+(?P<year_first>\d{2,4}))\b",
    flags=re.I,
)

def _date_parts(match):
    day = int(match.group("day") or match.group("day_first"))
    month_text = match.group("month") or match.group("month_first")
    year = int(match.group("year") or match.group("year_first"))
    return day, _MONTH_ALIASES[month_text.casefold()], year

def correct_unique_source_date(source_text, summary):
    """Replace one clearly mismatched full date only when each text has exactly one."""
    source_dates = list(_DATE_PATTERN.finditer(source_text))
    summary_dates = list(_DATE_PATTERN.finditer(summary))
    if len(source_dates) != 1 or len(summary_dates) != 1:
        return summary, None
    source_match, summary_match = source_dates[0], summary_dates[0]
    if _date_parts(source_match) == _date_parts(summary_match):
        return summary, None
    source_spelling = source_match.group(0)
    corrected = summary[:summary_match.start()] + source_spelling + summary[summary_match.end():]
    return corrected, {"generated_date": summary_match.group(0), "source_date": source_spelling}

def unsupported_full_dates(source_text, summary):
    """Return any complete summary dates that do not occur in the source."""
    source_dates = {_date_parts(match) for match in _DATE_PATTERN.finditer(source_text)}
    return sorted({
        match.group(0)
        for match in _DATE_PATTERN.finditer(summary)
        if _date_parts(match) not in source_dates
    })

def factuality_summary(claims):
    """Describe evidence outcomes without presenting model judgments as probabilities."""
    if not claims:
        return "No factual claims detected — no factuality conclusion can be made."
    statuses = {claim["assessment"] for claim in claims}
    if "checker_mismatch" in statuses:
        return "Claim checker added number/date details absent from the summary — its claim results are unreliable."
    if "evidence_not_verified" in statuses:
        return "Evidence not verified — at least one evidence quote was not found verbatim in the source."
    if "no_evidence_found" in statuses:
        return "No factual evidence detected for at least one claim."
    if "contradicted" in statuses:
        return "Potential contradiction detected — inspect the cited source text."
    if "not_found" in statuses:
        return "No supporting evidence located for at least one claim."
    if "unclear" in statuses:
        return "Evidence is unclear for at least one claim — manual review is needed."
    return "A source quote was found for every claim; the AI support judgments still need review."


## 3. Paste a source and run the evaluation

Paste a complete Roman-script Hinglish passage. Inputs longer than 2,048 model tokens are truncated at the end and clearly flagged. Claim checking uses the same model-visible text as summarization.

In [ ]:
source_box = widgets.Textarea(
    value="",
    placeholder="Paste Roman-script Hinglish here, such as Hindi words written in Latin letters mixed with English.",
    description="Source text",
    layout=widgets.Layout(width="100%", height="220px"),
    style={"description_width": "initial"}
)
model_picker = widgets.Dropdown(
    options=list(MODEL_IDS.keys()), value="Qwen3 1.7B",
    description="Model", style={"description_width": "initial"}
)
prompt_picker = widgets.Dropdown(
    options=["Factuality-aware", "Standard"], value="Factuality-aware",
    description="Prompt", style={"description_width": "initial"}
)
run_button = widgets.Button(
    description="Summarize and check claims", button_style="success", icon="play",
    layout=widgets.Layout(width="260px")
)
download_button = widgets.Button(
    description="Download last result (JSON)", icon="download", disabled=True
)
progress_bar = widgets.IntProgress(
    value=0, min=0, max=100, description="Progress", bar_style="info",
    layout=widgets.Layout(width="100%")
)
status_display = widgets.HTML(
    value="<b>Ready.</b> Paste Roman-script Hinglish and click the button to start."
)
result_output = widgets.Output()

def set_run_status(message, percent, state="info"):
    colors = {"info": "#1f6feb", "success": "#16803c", "warning": "#9a6700", "error": "#cf222e"}
    color = colors.get(state, colors["info"])
    status_display.value = (
        f'<div style="padding:8px 10px;border-left:4px solid {color};background:#f6f8fa">'
        f'<b style="color:{color}">{message}</b></div>'
    )
    progress_bar.value = max(0, min(100, int(percent)))
    progress_bar.bar_style = {"info": "info", "success": "success", "warning": "warning", "error": "danger"}.get(state, "info")

def on_run_clicked(_):
    global _last_result
    run_button.disabled = True
    download_button.disabled = True
    _last_result = None
    progress_bar.value = 1
    set_run_status("Starting… this tracker continues after the Colab cell’s green tick.", 1)
    with result_output:
        clear_output(wait=True)
        source = source_box.value.strip()
        if not source:
            display(Markdown("**Paste source text first.**"))
            set_run_status("Waiting for source text.", 0, "warning")
            run_button.disabled = False
            return
        if re.search(r"[\u0900-\u097F]", source):
            display(Markdown("**This demo expects Roman-script Hinglish.** Hindi words should be written in Latin letters, mixed with English."))
            set_run_status("Input needs correction: use Roman-script Hinglish.", 0, "warning")
            run_button.disabled = False
            return
        if not re.search(r"[A-Za-z]", source):
            display(Markdown("**No Latin-script text found.** Paste a Roman-script Hinglish source."))
            set_run_status("Input needs correction: no Latin-script text found.", 0, "warning")
            run_button.disabled = False
            return
        try:
            selected_model = model_picker.value
            selected_prompt = prompt_picker.value
            set_run_status("Loading model (first use downloads model weights)…", 10)
            model, tokenizer = load_model(selected_model)
            source_used, original_tokens, source_truncated = prepare_source(source, tokenizer)
            set_run_status("Generating English summary…", 35)
            summary, summary_tokens, summary_seconds, summary_cap_hit = make_summary(
                source_used, selected_prompt, model, tokenizer
            )
            if not summary:
                raise ValueError("The model returned an empty summary. Try the other model size.")
            _, initial_date_correction = correct_unique_source_date(source_used, summary)
            summary_numeric_mismatches = unsupported_numeric_tokens(source_used, summary)
            summary_date_mismatches = unsupported_full_dates(source_used, summary)
            summary_numeric_retry_used = False
            summary_numeric_retry_error = None
            if summary_numeric_mismatches or summary_date_mismatches or initial_date_correction:
                summary_numeric_retry_used = True
                set_run_status("Number/date mismatch found; retrying the summary once…", 50, "warning")
                try:
                    retry_summary, retry_tokens, retry_seconds, retry_cap_hit = make_summary(
                        source_used, selected_prompt, model, tokenizer, numeric_retry=True
                    )
                    if retry_summary:
                        summary = retry_summary
                        summary_tokens += retry_tokens
                        summary_seconds += retry_seconds
                        summary_cap_hit = retry_cap_hit
                except Exception as retry_error:
                    # Keep the original draft visible; the final guards below still apply.
                    summary_numeric_retry_error = f"{type(retry_error).__name__}: {retry_error}"
            summary, summary_date_correction = correct_unique_source_date(source_used, summary)
            summary_numeric_mismatches = unsupported_numeric_tokens(source_used, summary)
            summary_date_mismatches = unsupported_full_dates(source_used, summary)
            set_run_status("Checking summary claims against the source…", 65)
            claim_check_error = None
            try:
                claims, audit_tokens, audit_seconds, raw_audit = check_claims(
                    source_used, summary, model, tokenizer
                )
            except Exception as error:
                # Preserve a usable summary, but never imply it passed a check.
                claims = []
                audit_tokens, audit_seconds = 0, 0.0
                raw_audit = None
                claim_check_error = f"{type(error).__name__}: {error}"
            if summary_numeric_mismatches:
                factuality_status = (
                    "Possible number/date mismatch — the summary contains numeric values "
                    "not present in the source; review before use."
                )
            elif claim_check_error:
                factuality_status = "Claim check unavailable — no factuality conclusion can be made."
            else:
                factuality_status = factuality_summary(claims)
            set_run_status("Preparing the result…", 90)
            _last_result = {
                "project": "FactIndic interactive research demo",
                "model": MODEL_IDS[selected_model],
                "prompt": selected_prompt,
                "model_visible_source_sha256": hashlib.sha256(source_used.encode("utf-8")).hexdigest(),
                "source_tokens_before_cap": original_tokens,
                "source_truncated": source_truncated,
                "summary": summary,
                "summary_numeric_values_not_in_source": summary_numeric_mismatches,
                "summary_dates_not_in_source": summary_date_mismatches,
                "summary_numeric_retry_used": summary_numeric_retry_used,
                "summary_numeric_retry_error": summary_numeric_retry_error,
                "summary_date_correction": summary_date_correction,
                "summary_tokens": summary_tokens,
                "summary_generation_seconds": round(summary_seconds, 3),
                "summary_token_cap_hit": summary_cap_hit,
                "claim_check_tokens": audit_tokens,
                "claim_check_seconds": round(audit_seconds, 3),
                "claims": claims,
                "claim_check_completed": claim_check_error is None,
                "claim_check_error": claim_check_error,
                "factuality_status": factuality_status,
                "quote_validation": "displayed evidence quotes checked against the model-visible source",
                "all_displayed_quotes_found": all((not c["source_evidence"]) or c["quote_found_in_model_input"] for c in claims)
            }
            summary_heading = (
                "### Draft summary — number/date mismatch remains"
                if summary_numeric_mismatches or summary_date_mismatches else "### English summary"
            )
            display(Markdown(summary_heading + "\n\n" + summary))
            if source_truncated:
                display(Markdown(f"**Input note:** {original_tokens} tokens were provided; the model saw only the first {MAX_SOURCE_TOKENS}."))
            if summary_cap_hit:
                display(Markdown("**Output note:** the summary reached its token limit. Inspect it for an incomplete ending."))
            if summary_date_correction:
                display(Markdown(
                    "**Date correction:** the model changed a single full date. The notebook replaced `"
                    + summary_date_correction["generated_date"] + "` with the unique source date `"
                    + summary_date_correction["source_date"] + "`. Verify that it refers to the same event."
                ))
            if summary_numeric_mismatches or summary_date_mismatches:
                mismatch_values = summary_numeric_mismatches + summary_date_mismatches
                display(Markdown(
                    "**Number/date check:** the summary contains value(s) not found in the source: "
                    + ", ".join(mismatch_values)
                    + (
                        ". The automatic retry could not complete. Do not use this draft without correcting it against the source."
                        if summary_numeric_retry_error else
                        ". The automatic retry did not resolve this mismatch. Do not use this draft without correcting it against the source."
                    )
                ))
            display(Markdown("### Factuality evidence status\n\n" + factuality_status))
            display(Markdown("### Claim-by-claim evidence\n\nQuote presence is checked against the model-visible source. This does not prove logical support; the AI assessment can still be wrong."))
            if claims:
                display(pd.DataFrame(claims))
            elif claim_check_error:
                display(Markdown(
                    "**The claim check could not be completed.** The summary is shown for inspection, "
                    "but it has not been factually checked. Try the button again or use Qwen3 1.7B.\n\n"
                    f"Diagnostic: `{claim_check_error}`"
                ))
            else:
                display(Markdown("No factual claims were returned for checking. No factuality conclusion is available."))
            display(Markdown(
                f"**Run details:** {selected_model} · {selected_prompt} prompt · "
                f"summary {summary_seconds:.2f}s · claim check {audit_seconds:.2f}s · "
                f"{len(claims)} claims · same model generated and checked the summary."
            ))
            if claim_check_error:
                set_run_status(
                    f"Run finished; claim check unavailable: {claim_check_error}", 100, "warning"
                )
            elif summary_numeric_mismatches or summary_date_mismatches:
                set_run_status("Run finished with a number/date mismatch. Do not use this draft yet.", 100, "warning")
            elif summary_date_correction:
                set_run_status("Run finished after a source date correction; factuality still needs review.", 100, "warning")
            elif not claims or any(c["assessment"] != "supported" for c in claims):
                set_run_status("Run finished; manual factuality review is required.", 100, "warning")
            else:
                set_run_status("AI evidence check finished. Review the quotes and judgments.", 100, "success")
            download_button.disabled = False
        except Exception as error:
            _last_result = None
            set_run_status(f"Run failed: {type(error).__name__}: {error}", progress_bar.value, "error")
            display(Markdown(f"**Could not complete the check:** `{type(error).__name__}: {error}`. Try a shorter source or the 1.7B model."))
        finally:
            run_button.disabled = False

def on_download_clicked(_):
    if not _last_result:
        return
    output_path = Path("/content/factindic_interactive_result.json")
    output_path.write_text(json.dumps(_last_result, ensure_ascii=False, indent=2), encoding="utf-8")
    files.download(str(output_path))

run_button.on_click(on_run_clicked)
download_button.on_click(on_download_clicked)
display(widgets.VBox([
    source_box, widgets.HBox([model_picker, prompt_picker]),
    widgets.HBox([run_button, download_button]),
    widgets.HTML("<b>Run progress</b>"), progress_bar, status_display, result_output
]))


## Reading the output

- **A source quote was found for every claim** means the AI reviewer supplied text that appears verbatim in the source. It does not prove the claim is correct.
- **No factual evidence detected** means the reviewer did not provide a checkable source quote for at least one claim. Treat that claim as unsupported by this run.
- **Evidence not verified** means a quote was supplied but could not be found in the source exactly; the claim is not verified.
- **Potential contradiction**, **No supporting evidence located**, or **Unclear** needs human inspection.
- If no factual claims are detected, the notebook says that no factuality conclusion can be made. It does not invent a probability score.

The progress tracker follows the button-triggered model work. The green tick beside a Colab code cell only indicates that the cell which created the controls has finished. The same model performs both summarization and checking, so use this as a live project demonstration and AI-assisted review aid.
